# Forecasting and generative models for finance

Machine Learning for Finance, Dr. Arun B Ayyar. This notebook produces every number on the slides.

All data is simulated by `build/make_data.py`. The two CSV files it writes are in `data/`.

Packages: `pip install torch statsmodels prophet scikit-learn pytorch-forecasting lightning`

The whole notebook takes about 25 minutes on a laptop CPU. The Temporal Fusion Transformer and the GAN are the slow cells.

# Time series forecasting for finance

Daily net outflow of a simulated bank, six segments, 2022 to 2025. Positive values are money leaving the bank.
Every model forecasts the next 14 days. The last 182 days are the test period.

In [1]:
import os, sys, urllib.request

if 'google.colab' in sys.modules:            # Colab ships torch, statsmodels and scikit-learn
    get_ipython().system('pip -q install prophet pytorch-forecasting lightning')

if not os.path.exists('data/cashflow.csv'):  # not run from the repo folder: fetch the two input files
    os.makedirs('data', exist_ok=True)
    for f in ('cashflow.csv', 'transactions.csv'):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/iitm-da/finance-ml/master/data/' + f, 'data/' + f)

In [2]:
import json, time, logging, warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings('ignore')

DATA = 'data/'
cf = pd.read_csv(DATA + 'cashflow.csv', parse_dates=['date'])
cf.head()

,date,segment,net_outflow,dow,is_holiday,is_working,is_payday,is_dd_day,since_payday,post_holiday,yr_sin,yr_cos,december
0,2022-01-01,retail_current,82.15,5,0,0,0,0,4,0,0.017202,0.999852,0.0
1,2022-01-02,retail_current,70.70,6,0,0,0,0,4,0,0.034398,0.999408,0.0
2,2022-01-03,retail_current,69.59,0,1,0,0,0,4,0,0.051584,0.998669,0.0
3,2022-01-04,retail_current,201.59,1,0,1,0,1,4,1,0.068755,0.997634,0.0
4,2022-01-05,retail_current,113.00,2,0,1,0,0,4,0,0.085906,0.996303,0.0


In [3]:
SEGS = list(cf.segment.unique())
HEAD = 'retail_current'          # the segment drawn on the slides
H, ENC = 14, 56                  # forecast 14 days from the previous 56

wide = cf.pivot(index='date', columns='segment', values='net_outflow')[SEGS]
Y = wide.values                  # (1461 days, 6 segments)
cal = cf[cf.segment == HEAD].reset_index(drop=True)
n = len(cal)

TEST0 = n - 182                  # first test day
VAL0 = TEST0 - 91                # first validation day
ORIGINS = [TEST0 + 14 * k for k in range(13)]   # first day of each 14-day forecast
print(n, 'days | train to', cal.date[VAL0 - 1].date(), '| validation to', cal.date[TEST0 - 1].date(),
      '| test', cal.date[TEST0].date(), 'to', cal.date[n - 1].date())

1461 days | train to 2025-04-02 | validation to 2025-07-02 | test 2025-07-03 to 2025-12-31


## Scoring

MAE in GBP million, and MASE: MAE divided by the MAE of the seasonal naive forecast on the training days.
MASE below 1 beats "same weekday last week".

In [4]:
SCALE = np.abs(Y[7:TEST0] - Y[:TEST0 - 7]).mean(axis=0)      # per segment

def score(F):
    """F: (13 origins, 14 days, 6 segments) forecasts -> MAE per segment, mean MASE"""
    A = np.stack([Y[o:o + H] for o in ORIGINS])
    mae = np.abs(F - A).mean(axis=(0, 1))
    return mae, float((mae / SCALE).mean())

ACT = np.stack([Y[o:o + H] for o in ORIGINS])
FC, TIME, RES = {}, {}, {}

## Seasonal naive

Each day is forecast with the last observed value for the same weekday.

In [5]:
def seasonal_naive(o):
    return np.stack([Y[o + h - 7 * (h // 7 + 1)] for h in range(H)])

FC['naive'] = np.stack([seasonal_naive(o) for o in ORIGINS])
RES['naive'] = score(FC['naive'])
print('seasonal naive  MASE %.3f' % RES['naive'][1])

seasonal naive  MASE 1.036


## ARIMA

One seasonal ARIMA per segment, fitted once on the days before the test period.
At each origin the fitted model is given the history up to that day and asked for 14 days.

In [6]:
t0 = time.time()
F = np.zeros((len(ORIGINS), H, len(SEGS)))
arima_fit = {}
for j, seg in enumerate(SEGS):
    fit = SARIMAX(Y[:TEST0, j], order=(2, 0, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False, maxiter=200)
    arima_fit[seg] = fit
    for i, o in enumerate(ORIGINS):
        F[i, :, j] = fit.apply(Y[:o, j]).forecast(H)
FC['arima'] = F
TIME['arima'] = time.time() - t0
RES['arima'] = score(F)
print('ARIMA(2,0,1)(0,1,1)7  MASE %.3f  %.0f s' % (RES['arima'][1], TIME['arima']))
print(arima_fit[HEAD].summary().tables[1])

ARIMA(2,0,1)(0,1,1)7  MASE 0.744  21 s
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.2397     72.522     -0.003      0.997    -142.381     141.901
ar.L2         -0.0022      0.590     -0.004      0.997      -1.158       1.154
ma.L1          0.2315     72.523      0.003      0.997    -141.912     142.375
ma.S.L7       -0.9561      0.009   -103.840      0.000      -0.974      -0.938
sigma2       646.0316     16.700     38.685      0.000     613.300     678.763


## ARIMA with calendar regressors

The same model with the calendar passed as regressors: bank holiday, payday, the three days after it,
direct-debit day, the first working day after a holiday, and the December ramp.

In [7]:
cal['sp1'] = (cal.since_payday == 1).astype(int)
cal['sp2'] = (cal.since_payday == 2).astype(int)
cal['sp3'] = (cal.since_payday == 3).astype(int)
XCOLS = ['is_holiday', 'is_payday', 'sp1', 'sp2', 'sp3', 'is_dd_day', 'post_holiday', 'december']
X = cal[XCOLS].values.astype(float)

t0 = time.time()
F = np.zeros((len(ORIGINS), H, len(SEGS)))
for j, seg in enumerate(SEGS):
    fit = SARIMAX(Y[:TEST0, j], exog=X[:TEST0], order=(2, 0, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False, maxiter=200)
    for i, o in enumerate(ORIGINS):
        F[i, :, j] = fit.apply(Y[:o, j], exog=X[:o]).forecast(H, exog=X[o:o + H])
FC['arimax'] = F
TIME['arimax'] = time.time() - t0
RES['arimax'] = score(F)
print('ARIMA + calendar  MASE %.3f  %.0f s' % (RES['arimax'][1], TIME['arimax']))

ARIMA + calendar  MASE 0.534  117 s


## Prophet

Trend, weekly and yearly seasonality, bank holidays, and the payday and direct-debit flags as regressors.
Prophet fits a curve against the date. It does not read yesterday's value.

In [8]:
from prophet import Prophet
logging.getLogger('cmdstanpy').disabled = True
logging.getLogger('prophet').setLevel(logging.ERROR)

hol = pd.DataFrame({'holiday': 'bank_holiday', 'ds': cal.date[cal.is_holiday == 1]})
PREG = ['is_payday', 'sp1', 'sp2', 'sp3', 'is_dd_day', 'post_holiday']

t0 = time.time()
F = np.zeros((len(ORIGINS), H, len(SEGS)))
prophet_parts = None
for j, seg in enumerate(SEGS):
    df = pd.DataFrame({'ds': cal.date, 'y': Y[:, j]})
    df[PREG] = cal[PREG]
    m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False, holidays=hol)
    for r in PREG:
        m.add_regressor(r)
    m.fit(df.iloc[:TEST0])
    pred = m.predict(df.drop(columns='y'))
    for i, o in enumerate(ORIGINS):
        F[i, :, j] = pred.yhat.values[o:o + H]
    if seg == HEAD:
        prophet_parts = pred
FC['prophet'] = F
TIME['prophet'] = time.time() - t0
RES['prophet'] = score(F)
print('Prophet  MASE %.3f  %.0f s' % (RES['prophet'][1], TIME['prophet']))

Importing plotly failed. Interactive plots will not work.


Prophet  MASE 0.552  4 s


## LSTM

One network for all six segments. The encoder reads 56 days of outflow and calendar.
The decoder reads the calendar of the next 14 days and writes one number per day.

In [9]:
import torch
import torch.nn as nn

dow1h = np.eye(7)[cal.dow.values]
KNOWN = np.hstack([dow1h, cal[['is_holiday', 'is_payday', 'sp1', 'sp2', 'sp3', 'is_dd_day', 'post_holiday',
                               'yr_sin', 'yr_cos', 'december']].values]).astype(np.float32)
MU, SD = Y[:VAL0].mean(axis=0), Y[:VAL0].std(axis=0)
Z = ((Y - MU) / SD).astype(np.float32)           # each segment scaled by its own training mean and sd

def windows(starts):
    xe, xd, yy, sid = [], [], [], []
    for j in range(len(SEGS)):
        for s in starts:
            xe.append(np.hstack([Z[s - ENC:s, j:j + 1], KNOWN[s - ENC:s]]))
            xd.append(KNOWN[s:s + H])
            yy.append(Z[s:s + H, j])
            sid.append(j)
    return [torch.tensor(np.array(a)) for a in (xe, xd, yy)] + [torch.tensor(sid)]

TR = windows(range(ENC, VAL0 - H + 1))
VA = windows(range(VAL0, TEST0 - H + 1))
TE = windows(ORIGINS)
print('training windows', len(TR[0]), '| validation', len(VA[0]), '| test', len(TE[0]))

training windows 6714 | validation 468 | test 78


In [10]:
class Seq2Seq(nn.Module):
    def __init__(self, n_known=KNOWN.shape[1], hidden=64, n_seg=len(SEGS)):
        super().__init__()
        self.emb = nn.Embedding(n_seg, 4)
        self.enc = nn.LSTM(1 + n_known + 4, hidden, batch_first=True)
        self.dec = nn.LSTM(n_known + 4, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, xe, xd, sid):
        e = self.emb(sid)
        _, state = self.enc(torch.cat([xe, e[:, None].expand(-1, xe.shape[1], -1)], -1))
        d, _ = self.dec(torch.cat([xd, e[:, None].expand(-1, xd.shape[1], -1)], -1), state)
        return self.out(d).squeeze(-1)

def train_lstm(seed, max_epochs=60, patience=8):
    torch.manual_seed(seed); np.random.seed(seed)
    net = Seq2Seq()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    lossf = nn.L1Loss()
    best, best_state, bad, hist = 1e9, None, 0, []
    for ep in range(max_epochs):
        net.train()
        perm = torch.randperm(len(TR[0]))
        tot = 0.0
        for k in range(0, len(perm), 128):
            b = perm[k:k + 128]
            opt.zero_grad()
            loss = lossf(net(TR[0][b], TR[1][b], TR[3][b]), TR[2][b])
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            tot += loss.item() * len(b)
        net.eval()
        with torch.no_grad():
            v = lossf(net(VA[0], VA[1], VA[3]), VA[2]).item()
        hist.append((tot / len(perm), v))
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.clone() for k, t in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    net.load_state_dict(best_state)
    return net, hist

def lstm_forecast(net):
    net.eval()
    with torch.no_grad():
        p = net(TE[0], TE[1], TE[3]).numpy()            # (6 segments x 13 origins, 14)
    p = p.reshape(len(SEGS), len(ORIGINS), H).transpose(1, 2, 0)
    return p * SD + MU

t0 = time.time()
lstm_runs = []
for seed in range(5):
    net, hist = train_lstm(seed)
    f = lstm_forecast(net)
    lstm_runs.append((score(f)[1], f, hist))
    print('seed', seed, 'epochs', len(hist), 'MASE %.3f' % lstm_runs[-1][0])
TIME['lstm'] = (time.time() - t0) / 5
FC['lstm'] = lstm_runs[0][1]
RES['lstm'] = score(FC['lstm'])
lstm_mase = np.array([r[0] for r in lstm_runs])
lstm_params = sum(p.numel() for p in net.parameters())
print('LSTM  MASE %.3f (seed 0) | mean %.3f sd %.3f over 5 seeds | %d weights | %.0f s per run'
      % (RES['lstm'][1], lstm_mase.mean(), lstm_mase.std(), lstm_params, TIME['lstm']))

seed 0 epochs 35 MASE 0.546


seed 1 epochs 38 MASE 0.541


seed 2 epochs 50 MASE 0.548


seed 3 epochs 34 MASE 0.582


seed 4 epochs 26 MASE 0.546
LSTM  MASE 0.546 (seed 0) | mean 0.552 sd 0.015 over 5 seeds | 44889 weights | 41 s per run


## Temporal Fusion Transformer

Same windows. The TFT is told which inputs are static (the segment), which are known for the future (the calendar)
and which are only observed in the past (the outflow). It returns the 10th, 50th and 90th percentile for each day.

In [11]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
logging.getLogger('lightning.pytorch').setLevel(logging.ERROR)
logging.getLogger('lightning').setLevel(logging.ERROR)

long = cf.copy()
long['time_idx'] = long.groupby('segment').cumcount()
for c in ['dow', 'since_payday']:
    long[c] = long[c].astype(str)
KNOWN_REALS = ['is_holiday', 'is_payday', 'is_dd_day', 'post_holiday', 'yr_sin', 'yr_cos', 'december']
long[KNOWN_REALS] = long[KNOWN_REALS].astype(float)

training = TimeSeriesDataSet(
    long[long.time_idx < VAL0],
    time_idx='time_idx', target='net_outflow', group_ids=['segment'],
    min_encoder_length=ENC, max_encoder_length=ENC,
    min_prediction_length=H, max_prediction_length=H,
    static_categoricals=['segment'],
    time_varying_known_categoricals=['dow', 'since_payday'],
    time_varying_known_reals=KNOWN_REALS,
    time_varying_unknown_reals=['net_outflow'],
    target_normalizer=GroupNormalizer(groups=['segment']),
)
validation = TimeSeriesDataSet.from_dataset(training, long[long.time_idx < TEST0],
                                            min_prediction_idx=VAL0, stop_randomization=True)
train_dl = training.to_dataloader(train=True, batch_size=128, num_workers=0)
val_dl = validation.to_dataloader(train=False, batch_size=512, num_workers=0)

In [12]:
pl.seed_everything(0, verbose=False)
QS = [0.1, 0.5, 0.9]
tft = TemporalFusionTransformer.from_dataset(
    training, learning_rate=0.005, hidden_size=32, attention_head_size=4, dropout=0.1,
    hidden_continuous_size=16, loss=QuantileLoss(QS), log_interval=0,
)
ckpt = ModelCheckpoint(monitor='val_loss', dirpath='tft_ckpt', filename='best', save_top_k=1)
trainer = pl.Trainer(max_epochs=60, accelerator='cpu', gradient_clip_val=0.1, logger=False,
                     enable_progress_bar=False, enable_model_summary=False,
                     callbacks=[EarlyStopping(monitor='val_loss', patience=8), ckpt])
t0 = time.time()
trainer.fit(tft, train_dl, val_dl)
TIME['tft'] = time.time() - t0
tft = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)
tft_params = sum(p.numel() for p in tft.parameters())
print('TFT  %d epochs | %d weights | %.0f s' % (trainer.current_epoch, tft_params, TIME['tft']))

TFT  14 epochs | 84383 weights | 260 s


In [13]:
def tft_at(o, mode='quantiles'):
    """forecast the 14 days starting at index o, for every segment"""
    sub = long[(long.time_idx >= o - ENC) & (long.time_idx < o + H)]
    ds = TimeSeriesDataSet.from_dataset(training, sub, predict=True, stop_randomization=True)
    dl = ds.to_dataloader(train=False, batch_size=64, num_workers=0)
    out = tft.predict(dl, mode=mode, return_index=True, return_x=(mode == 'raw'),
                      trainer_kwargs=dict(logger=False, enable_progress_bar=False, accelerator='cpu'))
    return out

Q = np.zeros((len(ORIGINS), H, len(SEGS), 3))
for i, o in enumerate(ORIGINS):
    out = tft_at(o)
    order = [list(out.index.segment).index(s) for s in SEGS]
    Q[i] = out.output.numpy()[order].transpose(1, 0, 2)
FC['tft'] = Q[..., 1]
RES['tft'] = score(FC['tft'])
cover = float(((ACT >= Q[..., 0]) & (ACT <= Q[..., 2])).mean())
above = float((ACT > Q[..., 2]).mean())
below = float((ACT < Q[..., 0]).mean())
print('TFT median  MASE %.3f | actual inside P10-P90 on %.1f%% of days, above P90 %.1f%%, below P10 %.1f%%'
      % (RES['tft'][1], cover * 100, above * 100, below * 100))

TFT median  MASE 0.533 | actual inside P10-P90 on 71.8% of days, above P90 12.3%, below P10 15.9%


## What the TFT used

Variable selection weights and the attention over the 56 encoder days, summed over the 13 test origins.

In [14]:
imp = {'static': 0, 'encoder': 0, 'decoder': 0, 'attention': 0}
for o in ORIGINS:
    raw = tft_at(o, mode='raw')
    it = tft.interpret_output(raw.output, reduction='sum')
    imp['static'] = imp['static'] + it['static_variables'].numpy()
    imp['encoder'] = imp['encoder'] + it['encoder_variables'].numpy()
    imp['decoder'] = imp['decoder'] + it['decoder_variables'].numpy()
    imp['attention'] = imp['attention'] + it['attention'].numpy()
enc_imp = dict(zip(tft.encoder_variables, (imp['encoder'] / imp['encoder'].sum() * 100).round(1)))
dec_imp = dict(zip(tft.decoder_variables, (imp['decoder'] / imp['decoder'].sum() * 100).round(1)))
attn = imp['attention'][:ENC] / imp['attention'][:ENC].sum()
print('encoder', enc_imp)
print('decoder', dec_imp)
print('attention, last 15 encoder days', attn[-15:].round(3))

encoder {'dow': np.float32(14.0), 'since_payday': np.float32(2.4), 'is_holiday': np.float32(2.5), 'is_payday': np.float32(8.2), 'is_dd_day': np.float32(2.4), 'post_holiday': np.float32(3.8), 'yr_sin': np.float32(4.8), 'yr_cos': np.float32(3.5), 'december': np.float32(2.9), 'net_outflow': np.float32(55.5)}
decoder {'dow': np.float32(22.2), 'since_payday': np.float32(3.3), 'is_holiday': np.float32(6.6), 'is_payday': np.float32(9.2), 'is_dd_day': np.float32(5.1), 'post_holiday': np.float32(19.8), 'yr_sin': np.float32(9.2), 'yr_cos': np.float32(20.3), 'december': np.float32(4.3)}
attention, last 15 encoder days [0.016 0.02  0.027 0.014 0.007 0.022 0.018 0.015 0.02  0.028 0.013 0.007
 0.02  0.017 0.016]


## All models

In [15]:
NAMES = {'naive': 'Seasonal naive', 'arima': 'ARIMA', 'prophet': 'Prophet', 'arimax': 'ARIMA + calendar',
         'lstm': 'LSTM', 'tft': 'TFT (median)'}
tab = pd.DataFrame({NAMES[k]: list(RES[k][0].round(2)) + [round(RES[k][1], 3)] for k in NAMES},
                   index=SEGS + ['mean MASE']).T
tab

,retail_current,retail_savings,sme,corporate,cards,mortgages,mean MASE
Seasonal naive,26.99,10.52,17.82,39.45,9.21,11.01,1.036
ARIMA,18.68,8.14,12.31,30.00,7.45,6.65,0.744
Prophet,11.88,5.91,9.60,23.16,6.88,4.01,0.552
ARIMA + calendar,11.80,5.96,9.36,22.33,5.74,4.43,0.534
LSTM,10.74,6.01,8.84,23.59,6.17,5.07,0.546
TFT (median),10.89,5.93,8.53,21.55,6.14,5.01,0.533


## Applications: from a forecast to a decision

A treasury desk holds cash against the outflow it expects. The four cells below use the forecasts already made,
with no new model: a backtest of the P90, the margin each model needs, the whole bank, and the second week.

In [16]:
# 1. backtest of the P90: it should be exceeded on 10% of days
from scipy.stats import binomtest
j = SEGS.index(HEAD)
n_days = ACT[:, :, 0].size
rows = []
for i, s in enumerate(SEGS):
    up = int((ACT[:, :, i] > Q[:, :, i, 2]).sum()); dn = int((ACT[:, :, i] < Q[:, :, i, 0]).sum())
    rows.append([s, up, round(up / n_days * 100, 1), dn, round(dn / n_days * 100, 1),
                 round(float(binomtest(up, n_days, 0.10).pvalue), 4)])
bt = pd.DataFrame(rows, columns=['segment', 'days above P90', '% above', 'days below P10', '% below', 'p-value against 10%'])
APP = {'bt': rows, 'n_days': n_days}
bt

,segment,days above P90,% above,days below P10,% below,p-value against 10%
0,retail_current,33,18.1,27,14.8,0.0007
1,retail_savings,18,9.9,32,17.6,1.0000
2,sme,7,3.8,41,22.5,0.0029
3,corporate,28,15.4,23,12.6,0.0250
4,cards,31,17.0,3,1.6,0.0041
5,mortgages,17,9.3,48,26.4,0.9015


In [17]:
# 2. the margin each model needs: hold forecast + margin, and choose the margin that leaves 9 of 182 days short
rows = []
ERR = {}
for k in FC:
    e = (ACT[:, :, j] - FC[k][:, :, j]).ravel()              # positive: more went out than forecast
    c = float(np.quantile(e, 0.95, method='higher'))
    rows.append([NAMES[k], round(c, 1), int((e > c).sum()), round(float(np.clip(c - e, 0, None).mean()), 1)])
    ERR[k] = e.round(1).tolist()
mg = pd.DataFrame(rows, columns=['model', 'margin, GBP m', 'days short', 'idle cash per day, GBP m'])
APP['margin'] = rows; APP['err'] = ERR; APP['mean_out'] = round(float(ACT[:, :, j].mean()), 1)
mg

,model,"margin, GBP m",days short,"idle cash per day, GBP m"
0,Seasonal naive,58.0,9,61.5
1,ARIMA,45.9,9,50.3
2,ARIMA + calendar,25.2,9,26.1
3,Prophet,22.2,9,27.2
4,LSTM,25.5,9,23.0
5,TFT (median),25.0,9,24.4


In [18]:
# 3. the whole bank: add the six segments
tot = ACT.sum(axis=2); p50 = Q[..., 1].sum(axis=2); p90 = Q[..., 2].sum(axis=2)
seg_mae = np.abs(FC['tft'] - ACT).mean(axis=(0, 1))
e_tot = (tot - p50).ravel()
need = float(np.quantile(e_tot, 0.90, method='higher'))
APP['bank'] = {'seg_mae': seg_mae.round(1).tolist(), 'sum_seg_mae': round(float(seg_mae.sum()), 1),
               'mae_total': round(float(np.abs(FC['tft'].sum(axis=2) - tot).mean()), 1),
               'seg_margin': (Q[..., 2] - Q[..., 1]).mean(axis=(0, 1)).round(1).tolist(),
               'sum_margin': round(float((p90 - p50).mean()), 1), 'above_sum_p90': int((tot > p90).sum()),
               'need_margin': round(need, 1), 'above_need': int((e_tot > need).sum()),
               'mean_total': round(float(tot.mean()), 1)}
print('TFT median: sum of the six segment MAEs %.1f | MAE of the bank total %.1f'
      % (APP['bank']['sum_seg_mae'], APP['bank']['mae_total']))
print('sum of six P90s: %.1f above the summed medians, exceeded on %d of %d days'
      % (APP['bank']['sum_margin'], APP['bank']['above_sum_p90'], n_days))
print('margin exceeded on 10%% of days at bank level: %.1f (%d days)' % (need, APP['bank']['above_need']))

TFT median: sum of the six segment MAEs 58.1 | MAE of the bank total 28.9
sum of six P90s: 76.4 above the summed medians, exceeded on 3 of 182 days
margin exceeded on 10% of days at bank level: 44.8 (18 days)


In [19]:
# 4. the second week against the first: each holds every weekday once
rows = []
for k in FC:
    a = np.abs(FC[k] - ACT) / SCALE
    rows.append([NAMES[k], round(float(a[:, :7].mean()), 3), round(float(a[:, 7:].mean()), 3)])
APP['week'] = rows
pd.DataFrame(rows, columns=['model', 'MASE, days 1 to 7', 'MASE, days 8 to 14'])

,model,"MASE, days 1 to 7","MASE, days 8 to 14"
0,Seasonal naive,1.000,1.072
1,ARIMA,0.713,0.775
2,ARIMA + calendar,0.503,0.564
3,Prophet,0.553,0.552
4,LSTM,0.553,0.540
5,TFT (median),0.534,0.532


## Applications: ATM cash

The same four steps on a second problem. `data/atm.csv` holds the cash withdrawn each day at 200 ATMs over two
years, in GBP thousand; 20 of the ATMs opened 90 days before the test period. Forecasts are issued once a week
for the next 7 days, and the last 91 days are the test.

In [20]:
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
atm = pd.read_csv('data/atm.csv', parse_dates=['date']).sort_values(['atm_id', 'date']).reset_index(drop=True)
A_TEST = 91
d0 = atm.date.min(); atm['di'] = (atm.date - d0).dt.days
ti = int(atm.di.max()) - A_TEST + 1                              # first test day
NEW = set(atm.groupby('atm_id').di.min().loc[lambda s: s > 0].index)
atm['kind_c'] = atm.kind.astype('category').cat.codes
atm['atm_c'] = atm.atm_id.astype('category').cat.codes
atm['origin'] = atm.di - ((atm.di - ti) % 7)                     # the weekly forecast this day belongs to
atm['h'] = atm.di - atm.origin + 1                               # 1 to 7 days ahead
g = atm.groupby('atm_id').withdrawn
atm['naive'] = g.shift(7)                                        # seasonal naive: the same weekday last week
atm['r28'] = g.transform(lambda s: s.shift(1).rolling(28, min_periods=14).mean())
lvl = atm.loc[atm.h == 1, ['atm_id', 'origin', 'r28']].rename(columns={'r28': 'level'})
atm = atm.merge(lvl, on=['atm_id', 'origin'], how='left').dropna(subset=['level', 'naive'])
a_tr, a_te = atm[atm.di < ti], atm[atm.di >= ti].copy()
print('%d ATMs, %d of them new | training rows %d, test rows %d | training days for a new ATM: %d'
      % (atm.atm_id.nunique(), len(NEW), len(a_tr), len(a_te), a_tr[a_tr.atm_id.isin(NEW)].groupby('atm_id').size().max()))

200 ATMs, 20 of them new | training rows 113540, test rows 18200 | training days for a new ATM: 70


In [21]:
# one shared model for all 200 ATMs, one fit per quantile; and one median model per ATM on its own history
FEAT = ['dow', 'is_holiday', 'is_payday', 'since_payday', 'december', 'h', 'level', 'kind_c', 'atm_c']
LOCAL = FEAT[:7]
QS = [50, 80, 90, 95, 99]
t0 = time.time()
for q in QS:
    mdl = HGB(loss='quantile', quantile=q / 100, max_iter=400, learning_rate=.06, random_state=0,
              categorical_features=[FEAT.index(c) for c in ('dow', 'kind_c', 'atm_c')]).fit(a_tr[FEAT], a_tr.withdrawn)
    a_te[f'q{q}'] = mdl.predict(a_te[FEAT])
t_shared = time.time() - t0
t0 = time.time()
a_te['local'] = np.nan
for a, part in a_tr.groupby('atm_id'):
    mm = HGB(loss='quantile', quantile=.5, max_iter=80, learning_rate=.1, random_state=0,
             categorical_features=[0]).fit(part[LOCAL], part.withdrawn)
    a_te.loc[a_te.atm_id == a, 'local'] = mm.predict(a_te.loc[a_te.atm_id == a, LOCAL])
t_local = time.time() - t0
a_scale = (a_tr.withdrawn - a_tr.naive).abs().groupby(a_tr.atm_id).mean()
a_te['sc'] = a_te.atm_id.map(a_scale)
is_new = a_te.atm_id.isin(NEW)
def a_mase(col, mask):
    d = a_te[mask]
    return float(((d.withdrawn - d[col]).abs() / d.sc).groupby(d.atm_id).mean().mean())
rows = [[nm, round(a_mase(c, ~is_new), 3), round(a_mase(c, is_new), 3), round(a_mase(c, is_new | ~is_new), 3)]
        for nm, c in [('Seasonal naive', 'naive'), ('One model per ATM', 'local'), ('One shared model', 'q50')]]
print('shared model: %.0f s for %d quantiles | 200 models, one per ATM: %.0f s' % (t_shared, len(QS), t_local))
pd.DataFrame(rows, columns=['MASE', '180 established ATMs', '20 new ATMs', 'all 200'])

shared model: 31 s for 5 quantiles | 200 models, one per ATM: 115 s


,MASE,180 established ATMs,20 new ATMs,all 200
0,Seasonal naive,0.941,1.004,0.947
1,One model per ATM,0.720,0.842,0.732
2,One shared model,0.644,0.758,0.656


In [22]:
# backtest of the P95, ATM by ATM: it should be exceeded on 5% of days
above95 = (a_te.withdrawn > a_te.q95)
per_atm = (above95.groupby(a_te.atm_id).sum()).astype(int)
print('all ATM-days: above the P95 on %.2f%% | by quantile: %s'
      % (above95.mean() * 100, ', '.join('P%d %.1f%%' % (q, (a_te.withdrawn > a_te[f"q{q}"]).mean() * 100) for q in QS)))
print('ATMs by number of days above the P95, out of 91:')
print(per_atm.value_counts().sort_index().to_string())
print('ATMs with 10 or more days above (binomial p < 0.02 for a true 5%%): %d' % int((per_atm >= 10).sum()))

all ATM-days: above the P95 on 4.23% | by quantile: P50 41.5%, P80 15.6%, P90 7.7%, P95 4.2%, P99 1.0%
ATMs by number of days above the P95, out of 91:
0      9
1     38
2     27
3     32
4     28
5     23
6     11
7     10
8      9
9      3
10     5
11     2
14     1
15     1
19     1
ATMs with 10 or more days above (binomial p < 0.02 for a true 5%): 10


In [23]:
# replenishment: every R days an ATM is topped up to the sum of the next R daily forecasts at one quantile
Dm = a_te.pivot(index='di', columns='atm_id', values='withdrawn').values
Fq = {q: a_te.pivot(index='di', columns='atm_id', values=f'q{q}').values for q in QS}
def replenish(q, R, keep=None):
    cash = np.zeros(Dm.shape[1]); out_days = unmet = idle = loaded = 0.0; tr_ = []
    for d in range(A_TEST):
        if d % R == 0:
            target = Fq[q][d:d + R].sum(axis=0)
            loaded += np.clip(target - cash, 0, None).sum(); cash = np.maximum(cash, target)
        start = cash.copy()
        short = np.clip(Dm[d] - cash, 0, None)
        out_days += (short > 0).sum(); unmet += short.sum()
        cash = np.clip(cash - Dm[d], 0, None); idle += cash.sum()
        if keep is not None:
            tr_.append([round(float(start[keep]), 1), round(float(cash[keep]), 1), int(short[keep] > 0)])
    n_ = Dm.size
    return {'out': round(out_days / n_ * 100, 2), 'unmet': round(unmet / Dm.sum() * 100, 2),
            'idle': round(idle / n_, 1), 'visits': round(7 / R, 2), 'trace': tr_}
ids = sorted(a_te.atm_id.unique())
KEEP = next(i for i, a in enumerate(ids) if a not in NEW and a_te.loc[a_te.atm_id == a, 'kind'].iloc[0] == 'residential')
GRID = {f'{q}_{R}': replenish(q, R, KEEP) for q in QS for R in (1, 2, 3, 7)}
pol = pd.DataFrame([[f'P{k.split("_")[0]}', int(k.split('_')[1]), v['out'], v['unmet'], v['idle'], v['visits']] for k, v in GRID.items()],
                   columns=['quantile', 'days between visits', '% ATM-days empty', '% of demand not served',
                            'cash left at day end, GBP k per ATM', 'visits per ATM per week'])
print('mean withdrawn per ATM per day in the test: %.1f' % Dm.mean())
pol

mean withdrawn per ATM per day in the test: 14.4


,quantile,days between visits,% ATM-days empty,% of demand not served,"cash left at day end, GBP k per ATM",visits per ATM per week
0,P50,1,41.38,8.95,1.7,7.00
1,P50,2,20.99,6.91,8.8,3.50
2,P50,3,14.66,6.01,16.4,2.33
3,P50,7,7.02,4.46,45.6,1.00
4,P80,1,15.42,2.69,4.1,7.00
5,P80,2,5.65,1.44,14.6,3.50
6,P80,3,3.12,0.92,25.4,2.33
7,P80,7,0.75,0.39,67.9,1.00
8,P90,1,7.60,1.31,5.8,7.00
9,P90,2,2.20,0.53,18.2,3.50


In [24]:
# kept for the slides
ex = {}
for kd in ['high_street', 'residential', 'transport']:
    a = next(a for a in ids if a not in NEW and atm.loc[atm.atm_id == a, 'kind'].iloc[0] == kd)
    p = atm[(atm.atm_id == a) & (atm.di >= ti - 42) & (atm.di < ti)]
    ex[kd] = {'id': a, 'y': p.withdrawn.round(1).tolist(), 'dow': p.dow.tolist(), 'pay': p.is_payday.tolist(),
              'hol': p.is_holiday.tolist(), 'dates': [str(x.date()) for x in p.date]}
APP['atm'] = {'n': len(ids), 'n_new': len(NEW), 'days': int(atm.di.max()) + 1, 'test': A_TEST, 'ex': ex,
              'kinds': a_te.groupby('kind').atm_id.nunique().to_dict(), 'mase': rows,
              't_shared': round(t_shared), 't_local': round(t_local),
              'above': {str(q): round(float((a_te.withdrawn > a_te[f'q{q}']).mean() * 100), 2) for q in QS},
              'per_atm': per_atm.tolist(), 'new_train_days': int(a_tr[a_tr.atm_id.isin(NEW)].groupby('atm_id').size().max()),
              'grid': GRID, 'keep': ids[KEEP], 'mean_day': round(float(Dm.mean()), 1)}
print('kept', list(APP['atm']))

kept ['n', 'n_new', 'days', 'test', 'ex', 'kinds', 'mase', 't_shared', 't_local', 'above', 'per_atm', 'new_train_days', 'grid', 'keep', 'mean_day']


# Generative models, applied

100,000 simulated card transactions over 180 days, 0.9% of them fraud.
Part 1 trains a GAN on them and tests the synthetic table three ways: fidelity, utility, privacy.
Part 2 builds the fraud detection pipeline on the same table.

In [25]:
import json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import ks_2samp
from sklearn.preprocessing import QuantileTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings('ignore')

DATA = 'data/'
tx = pd.read_csv(DATA + 'transactions.csv')
tx.head()

,day,hour,amount,category,channel,customer_age,tenure_months,distance_km,txns_last_24h,is_fraud
0,0,14,36.74,grocery,contactless,70,55,15.2,2,0
1,0,19,8.88,restaurant,contactless,48,27,2.3,3,0
2,0,12,61.47,online_retail,online,39,77,0.0,0,0
3,0,18,22.73,grocery,chip_pin,68,200,2.2,1,0
4,0,9,27.55,grocery,contactless,80,26,7.3,2,0


In [26]:
NUM = ['hour', 'amount', 'customer_age', 'tenure_months', 'distance_km', 'txns_last_24h']
CAT = ['category', 'channel']
CATS = {c: sorted(tx[c].unique()) for c in CAT}

train = tx[tx.day < 120].reset_index(drop=True)        # first 120 days
test = tx[tx.day >= 120].reset_index(drop=True)        # last 60 days
hold = train.sample(frac=0.2, random_state=0)          # real rows the GAN does not see
fit = train.drop(hold.index).reset_index(drop=True)    # rows the GAN is trained on
hold = hold.reset_index(drop=True)
print('GAN training rows', len(fit), '| held out', len(hold), '| test', len(test))
print('fraud rate  fit %.2f%%  test %.2f%%' % (fit.is_fraud.mean() * 100, test.is_fraud.mean() * 100))

GAN training rows 53148 | held out 13287 | test 33565
fraud rate  fit 0.90%  test 0.84%


## Encoding

Numeric columns go through a quantile transform to a normal shape. Category and channel are one-hot.
The fraud flag is the condition: the generator is told which kind of row to write.

In [27]:
qt = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=0).fit(fit[NUM])

def encode(df):
    parts = [qt.transform(df[NUM])]
    for c in CAT:
        parts.append(np.eye(len(CATS[c]))[pd.Categorical(df[c], categories=CATS[c]).codes])
    return np.hstack(parts).astype(np.float32)

def decode(a, cond):
    out = pd.DataFrame(qt.inverse_transform(a[:, :len(NUM)]), columns=NUM)
    k = len(NUM)
    for c in CAT:
        m = len(CATS[c])
        out[c] = np.array(CATS[c])[a[:, k:k + m].argmax(1)]
        k += m
    for c in ['hour', 'customer_age', 'tenure_months', 'txns_last_24h']:
        out[c] = out[c].round().astype(int)
    out['amount'] = out.amount.round(2)
    out['distance_km'] = out.distance_km.round(1)
    out['is_fraud'] = cond.astype(int)
    return out[NUM + CAT + ['is_fraud']]

Xfit = torch.tensor(encode(fit))
Cfit = torch.tensor(fit.is_fraud.values[:, None].astype(np.float32))
D_X, D_Z = Xfit.shape[1], 64
BLOCKS = [len(CATS[c]) for c in CAT]
print('encoded width', D_X)

encoded width 17


## The two networks

In [28]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D_Z + 1, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                                 nn.Linear(256, D_X))

    def forward(self, z, c, tau=0.2):
        h = self.net(torch.cat([z, c], 1))
        out, k = [h[:, :len(NUM)]], len(NUM)
        for m in BLOCKS:                           # one soft one-hot per categorical column
            out.append(nn.functional.gumbel_softmax(h[:, k:k + m], tau=tau))
            k += m
        return torch.cat(out, 1)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(D_X + 1, 256), nn.LeakyReLU(0.2), nn.Linear(256, 256),
                                 nn.LeakyReLU(0.2), nn.Linear(256, 1))

    def forward(self, x, c):
        return self.net(torch.cat([x, c], 1))

def sample(G, n, fraud_rate=None, cond=None, seed=0):
    g = torch.Generator().manual_seed(seed)
    if cond is None:
        cond = (torch.rand(n, 1, generator=g) < fraud_rate).float()
    z = torch.randn(n, D_Z, generator=g)
    G.eval()
    with torch.no_grad():
        a = G(z, cond).numpy()
    G.train()
    return decode(a, cond.numpy()[:, 0])

## Training

Wasserstein loss with a gradient penalty. The critic takes five steps for each generator step.
Fraud rows are drawn into every batch at 20% so the generator sees enough of them.

In [29]:
torch.manual_seed(0); np.random.seed(0)
G, C = Generator(), Critic()
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.9))
optC = torch.optim.Adam(C.parameters(), lr=2e-4, betas=(0.5, 0.9))
BATCH, N_CRITIC, EPOCHS = 512, 5, 150
STEPS = len(fit) // BATCH
idx_f = np.where(fit.is_fraud.values == 1)[0]
idx_l = np.where(fit.is_fraud.values == 0)[0]
RATE = float(fit.is_fraud.mean())
SNAP_AT = [0, 1, 2, 3, 5, 8, 12, 20, 30, 50, 80, 110, 150]
snaps, wdist = [], []

def snapshot(ep):
    s = sample(G, 4000, fraud_rate=RATE, seed=1)
    ks = {c: float(ks_2samp(s[c], fit[c]).statistic) for c in NUM}
    snaps.append({'epoch': ep, 'ks': ks, 'amount': s.amount.values[:600].tolist(), 'hour': s.hour.values[:600].tolist()})
    print('epoch %3d  KS amount %.3f  hour %.3f  mean %.3f' % (ep, ks['amount'], ks['hour'], np.mean(list(ks.values()))))

t0 = time.time()
snapshot(0)
for ep in range(1, EPOCHS + 1):
    w = 0.0
    for step in range(STEPS):
        nf = BATCH // 5
        b = np.concatenate([np.random.choice(idx_f, nf), np.random.choice(idx_l, BATCH - nf)])
        x, c = Xfit[b], Cfit[b]
        fake = G(torch.randn(BATCH, D_Z), c).detach()
        eps = torch.rand(BATCH, 1)
        mix = (eps * x + (1 - eps) * fake).requires_grad_(True)
        grad = torch.autograd.grad(C(mix, c).sum(), mix, create_graph=True)[0]
        gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
        gap = C(x, c).mean() - C(fake, c).mean()          # the critic's estimate of the distance
        lossC = -gap + 10 * gp
        optC.zero_grad(); lossC.backward(); optC.step()
        w += gap.item()
        if step % N_CRITIC == 0:
            lossG = -C(G(torch.randn(BATCH, D_Z), c), c).mean()
            optG.zero_grad(); lossG.backward(); optG.step()
    wdist.append(w / STEPS)
    if ep in SNAP_AT:
        snapshot(ep)
GAN_TIME = time.time() - t0
print('trained in %.0f s' % GAN_TIME)

epoch   0  KS amount 0.423  hour 0.404  mean 0.400


epoch   1  KS amount 0.438  hour 0.600  mean 0.550


epoch   2  KS amount 0.432  hour 0.488  mean 0.509


epoch   3  KS amount 0.487  hour 0.414  mean 0.503


epoch   5  KS amount 0.406  hour 0.398  mean 0.461


epoch   8  KS amount 0.104  hour 0.205  mean 0.200


... (3 outputs left out)

epoch  50  KS amount 0.038  hour 0.069  mean 0.080


epoch  80  KS amount 0.015  hour 0.042  mean 0.072


epoch 110  KS amount 0.014  hour 0.031  mean 0.053


epoch 150  KS amount 0.017  hour 0.044  mean 0.053
trained in 318 s


In [30]:
synth = sample(G, len(fit), fraud_rate=RATE, seed=2)
synth.head()

,hour,amount,customer_age,tenure_months,distance_km,txns_last_24h,category,channel,is_fraud
0,23,47.560001,53,1,0.0,0,online_retail,online,0
1,21,23.059999,53,34,13.5,2,restaurant,chip_pin,0
2,15,46.500000,52,77,23.6,2,grocery,contactless,0
3,17,54.869999,61,45,2.2,1,cash_withdrawal,atm,0
4,10,11.740000,54,15,0.0,3,online_retail,online,0


## Test 1: fidelity

Column by column: the KS statistic for numeric columns, total variation distance for categorical ones.
The held-out real rows give the value two real samples produce.

In [31]:
def tvd(a, b):
    p, q = a.value_counts(normalize=True), b.value_counts(normalize=True)
    i = p.index.union(q.index)
    return float((p.reindex(i, fill_value=0) - q.reindex(i, fill_value=0)).abs().sum() / 2)

fid = pd.DataFrame({
    'synthetic vs real': {**{c: ks_2samp(synth[c], fit[c]).statistic for c in NUM}, **{c: tvd(synth[c], fit[c]) for c in CAT}},
    'held-out real vs real': {**{c: ks_2samp(hold[c], fit[c]).statistic for c in NUM}, **{c: tvd(hold[c], fit[c]) for c in CAT}},
}).round(3)
fid

,synthetic vs real,held-out real vs real
hour,0.041,0.004
amount,0.016,0.011
customer_age,0.047,0.008
tenure_months,0.018,0.007
distance_km,0.102,0.010
txns_last_24h,0.109,0.007
category,0.078,0.007
channel,0.044,0.002


In [32]:
# pairwise structure: Spearman correlations of the numeric columns and the fraud flag
cols = NUM + ['is_fraud']
cr, cs = fit[cols].corr('spearman').values, synth[cols].corr('spearman').values
iu = np.triu_indices(len(cols), 1)
corr_diff = np.abs(cr - cs)[iu]
print('correlation gap: mean %.3f, largest %.3f' % (corr_diff.mean(), corr_diff.max()))

correlation gap: mean 0.012, largest 0.028


In [33]:
# rules that hold in every real row
def rules(df):
    return pd.Series({
        'online_retail not on the online channel': ((df.category == 'online_retail') & (df.channel != 'online')).mean(),
        'cash_withdrawal not at an ATM': ((df.category == 'cash_withdrawal') & (df.channel != 'atm')).mean(),
        'online with a distance above 0': ((df.channel == 'online') & (df.distance_km > 0)).mean(),
    })
rule_tab = (pd.DataFrame({'real': rules(fit), 'synthetic': rules(synth)}) * 100).round(2)
rule_tab

,real,synthetic
online_retail not on the online channel,0.0,0.69
cash_withdrawal not at an ATM,0.0,2.96
online with a distance above 0,0.0,0.00


## Test 2: utility

Train a fraud classifier on the synthetic table, test it on the real last 60 days, and compare with the same
classifier trained on the real rows.

In [34]:
def features(df):
    X = df[NUM].copy()
    for c in CAT:
        X[c] = pd.Categorical(df[c], categories=CATS[c]).codes
    return X

def fraud_model(df, seed=0, **kw):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06, random_state=seed,
                                          categorical_features=[len(NUM), len(NUM) + 1], **kw).fit(features(df), df.is_fraud)

def evaluate(model):
    p = model.predict_proba(features(test))[:, 1]
    return roc_auc_score(test.is_fraud, p), average_precision_score(test.is_fraud, p)

trtr = evaluate(fraud_model(fit))
tstr = evaluate(fraud_model(synth))
print('fraud rate  real %.2f%%  synthetic %.2f%%' % (fit.is_fraud.mean() * 100, synth.is_fraud.mean() * 100))
print('train on real,      test on real: ROC-AUC %.3f  PR-AUC %.3f' % trtr)
print('train on synthetic, test on real: ROC-AUC %.3f  PR-AUC %.3f' % tstr)

fraud rate  real 0.90%  synthetic 0.88%
train on real,      test on real: ROC-AUC 0.904  PR-AUC 0.346
train on synthetic, test on real: ROC-AUC 0.884  PR-AUC 0.235


## Test 3: privacy

Distance to the closest record. For each synthetic row, find the nearest training row.
Held-out real rows give the distances a table with no copying produces.
A copy of the training rows with a little noise added is the generator this test has to catch.

In [35]:
def dcr(a, ref):
    nn_ = NearestNeighbors(n_neighbors=1).fit(ref)
    return nn_.kneighbors(a)[0][:, 0]

E_fit, E_hold, E_syn = encode(fit), encode(hold), encode(synth.iloc[:len(hold)])
rng = np.random.default_rng(0)
pick = rng.choice(len(E_fit), len(hold), replace=False)
E_copy = E_fit[pick].copy()
E_copy[:, :len(NUM)] += rng.normal(0, 0.05, (len(E_copy), len(NUM)))

d_hold, d_syn, d_copy = dcr(E_hold, E_fit), dcr(E_syn, E_fit), dcr(E_copy, E_fit)
p5 = np.percentile(d_hold, 5)
for name, d in [('held-out real', d_hold), ('GAN synthetic', d_syn), ('noisy copy', d_copy)]:
    print('%-14s median distance %.3f | closer than the 5th percentile of held-out: %.1f%%'
          % (name, np.median(d), (d < p5).mean() * 100))

held-out real  median distance 0.489 | closer than the 5th percentile of held-out: 5.0%
GAN synthetic  median distance 0.535 | closer than the 5th percentile of held-out: 5.6%
noisy copy     median distance 0.115 | closer than the 5th percentile of held-out: 99.7%


In [36]:
# which is nearer: a training row, or a real row the generator did not see?
E_fit_sub = E_fit[pick]                       # as many training rows as held-out rows
def nearer_train(a):
    return float((dcr(a, E_fit_sub) < dcr(a, E_hold)).mean())
share_syn, share_copy = nearer_train(E_syn), nearer_train(E_copy)
print('nearer to a training row than to a held-out row: GAN %.1f%%, noisy copy %.1f%% (50%% means no memorising)'
      % (share_syn * 100, share_copy * 100))

exact = len(synth.merge(fit[NUM + CAT].drop_duplicates(), on=NUM + CAT))
print('synthetic rows identical to a training row:', exact, 'of', len(synth))

nearer to a training row than to a held-out row: GAN 50.2%, noisy copy 99.9% (50% means no memorising)
synthetic rows identical to a training row: 0 of 53148


In [37]:
# the rare class: 600 synthetic fraud rows against the training fraud rows
syn_f = sample(G, 600, cond=torch.ones(600, 1), seed=3)
Ef_fit, Ef_hold, Ef_syn = encode(fit[fit.is_fraud == 1]), encode(hold[hold.is_fraud == 1]), encode(syn_f)
df_hold, df_syn = dcr(Ef_hold, Ef_fit), dcr(Ef_syn, Ef_fit)
print('fraud rows only: median distance to the nearest training fraud row: held-out %.3f, synthetic %.3f'
      % (np.median(df_hold), np.median(df_syn)))
print('training fraud rows', len(Ef_fit), '| held-out fraud rows', len(Ef_hold))

fraud rows only: median distance to the nearest training fraud row: held-out 1.088, synthetic 1.272
training fraud rows 477 | held-out fraud rows 120


# Fraud detection pipeline

Train on the first 120 days, score the last 60.

In [38]:
n_test, n_fraud = len(test), int(test.is_fraud.sum())
print('test rows', n_test, '| fraud', n_fraud, '| accuracy of flagging nothing %.2f%%' % ((1 - n_fraud / n_test) * 100))

test rows 33565 | fraud 281 | accuracy of flagging nothing 99.16%


In [39]:
# without labels: isolation forest scores how easy a row is to isolate
iso = IsolationForest(n_estimators=300, random_state=0).fit(pd.get_dummies(features(train), columns=CAT))
s_iso = -iso.score_samples(pd.get_dummies(features(test), columns=CAT))
# with labels: gradient boosting
gb = fraud_model(train)
s_gb = gb.predict_proba(features(test))[:, 1]
for name, s in [('isolation forest', s_iso), ('gradient boosting', s_gb)]:
    print('%-18s ROC-AUC %.3f  PR-AUC %.3f' % (name, roc_auc_score(test.is_fraud, s), average_precision_score(test.is_fraud, s)))

isolation forest   ROC-AUC 0.772  PR-AUC 0.049
gradient boosting  ROC-AUC 0.908  PR-AUC 0.361


In [40]:
# alerts per day -> what the analysts catch
def at_budget(score, per_day, days=60):
    k = per_day * days
    top = np.argsort(-score)[:k]
    tp = int(test.is_fraud.values[top].sum())
    return {'alerts': k, 'caught': tp, 'precision': tp / k, 'recall': tp / n_fraud,
            'gbp': float(test.amount.values[top][test.is_fraud.values[top] == 1].sum())}
for per_day in [5, 10, 20, 50]:
    a, b = at_budget(s_iso, per_day), at_budget(s_gb, per_day)
    print('%2d alerts a day: isolation forest catches %3d of %d (precision %.0f%%) | gradient boosting %3d (precision %.0f%%)'
          % (per_day, a['caught'], n_fraud, a['precision'] * 100, b['caught'], b['precision'] * 100))

 5 alerts a day: isolation forest catches  32 of 281 (precision 11%) | gradient boosting 110 (precision 37%)
10 alerts a day: isolation forest catches  52 of 281 (precision 9%) | gradient boosting 138 (precision 23%)
20 alerts a day: isolation forest catches  70 of 281 (precision 6%) | gradient boosting 159 (precision 13%)
50 alerts a day: isolation forest catches 123 of 281 (precision 4%) | gradient boosting 200 (precision 7%)


## Does synthetic fraud help the classifier?

Add k GAN fraud rows to the training set and compare with repeating real fraud rows and with class weights.

In [41]:
def pr(df, seed=0, **kw):
    return evaluate(fraud_model(df, seed, **kw))[1]

aug = {}
real_f = train[train.is_fraud == 1]
for k in [0, 500, 2000, 5000]:
    g_, o_ = [], []
    for seed in range(5):
        if k == 0:
            g_.append(pr(train, seed)); o_.append(g_[-1]); continue
        sf = sample(G, k, cond=torch.ones(k, 1), seed=100 + seed)
        g_.append(pr(pd.concat([train, sf], ignore_index=True), seed))
        o_.append(pr(pd.concat([train, real_f.sample(k, replace=True, random_state=seed)], ignore_index=True), seed))
    aug[k] = (float(np.mean(g_)), float(np.std(g_)), float(np.mean(o_)), float(np.std(o_)))
    print('k=%4d  + GAN fraud rows: PR-AUC %.3f (sd %.3f) | + repeated real fraud rows: %.3f (sd %.3f)' % ((k,) + aug[k]))
cw = [pr(train, seed, class_weight='balanced') for seed in range(5)]
print('class_weight=balanced: PR-AUC %.3f' % np.mean(cw))

k=   0  + GAN fraud rows: PR-AUC 0.359 (sd 0.010) | + repeated real fraud rows: 0.359 (sd 0.010)


k= 500  + GAN fraud rows: PR-AUC 0.367 (sd 0.008) | + repeated real fraud rows: 0.364 (sd 0.007)


k=2000  + GAN fraud rows: PR-AUC 0.365 (sd 0.004) | + repeated real fraud rows: 0.374 (sd 0.006)


k=5000  + GAN fraud rows: PR-AUC 0.356 (sd 0.006) | + repeated real fraud rows: 0.377 (sd 0.005)


class_weight=balanced: PR-AUC 0.327


In [42]:
# drift check: population stability index of the model score, first 30 test days against the last 30
def psi(a, b, bins=10):
    cuts = np.quantile(a, np.linspace(0, 1, bins + 1)[1:-1])
    pa = np.bincount(np.searchsorted(cuts, a), minlength=bins) / len(a)
    pb = np.bincount(np.searchsorted(cuts, b), minlength=bins) / len(b)
    return float(((pa - pb) * np.log((pa + 1e-6) / (pb + 1e-6))).sum())
first, last = (test.day < 150).values, (test.day >= 150).values
psi_score = psi(s_gb[first], s_gb[last])
print('PSI of the score, days 120-149 against 150-179: %.4f (above 0.2 is the usual alarm)' % psi_score)

PSI of the score, days 120-149 against 150-179: 0.0012 (above 0.2 is the usual alarm)
